## Analyse 1: Häufigste Parking Violation Typen FY2023–FY2025
Welche Violation-Typen kommen am häufigsten vor und wie verändern sie sich über die Jahre?

In [ ]:
import pyspark.sql.functions as f
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
from pyspark.sql import SparkSession

# Alte (evtl. tote) Session sauber wegräumen
try:
    SparkSession.builder.getOrCreate().stop()
except Exception:
    pass

# Auch den globalen aktiven Context auf None setzen, damit getOrCreate() wirklich neu baut
from pyspark import SparkContext
SparkContext._active_spark_context = None

# Jetzt frisch aufbauen
spark = SparkSession.builder \
    .appName("BDLC_Parking_Violations_ViolationCode") \
    .master("spark://bdlc-012.bdlc.ls.eee.intern:7077") \
    .config("spark.cores.max", "12") \
    .getOrCreate()


spark.sparkContext.setLogLevel("WARN")

# Kontrolle: was hat diese Session tatsächlich bekommen?
cconf = spark.sparkContext.getConf()
print("cores.max     :", cconf.get("spark.cores.max"))
print("executor.cores:", cconf.get("spark.executor.cores", "(Default)"))
print("executor.mem  :", cconf.get("spark.executor.memory", "(Default)"))

# Verifizieren, dass der Context wirklich lebt
print("SparkContext active:", not spark.sparkContext._jsc.sc().isStopped())
spark

In [ ]:
%%time
processed_path = "hdfs:///parking_violations/processed/parking_violations_cleaned_v5"
df_raw = spark.read.parquet(processed_path)

# Analysebasis: nur vollständig erfasste Fiskaljahre FY2023–FY2025.
# Stray-Zeilen FY2022 (306'279) und FY2026 (882) werden ausgeschlossen,
# damit Ranking und Jahresvergleich auf exakt derselben Basis stehen.
df = (df_raw.filter("is_complete_fy")
            .select("fy", "violation_code", "violation_description_official")
            .cache())

print("Analysebasis (FY2023–FY2025):", df.count(), "Zeilen")

In [ ]:
#komplettes Schema
df_raw.printSchema()

In [ ]:
#reduziertes für diese Frage relevantes Schema
df.printSchema()

### Übersicht: Violations pro Jahr und Code

In [ ]:
%%time
violation_by_year = df.groupBy("fy", "violation_code") \
    .agg(
        f.count("*").alias("count"),
        f.first("violation_description_official").alias("description")
    ) \
    .orderBy("fy", f.desc("count"))

violation_by_year.show(20, truncate=False)

### Resultate
Violation Code 36 (Schulzone Tempolimit) dominiert mit über 8 Mio. Verstössen in FY2023 
deutlich. Code 21 (Parkverbot Strassenreinigung) und Code 38 (Parkuhr) folgen auf 
Platz 2 und 3.

### Top 10 Violation Codes gesamt (FY2023–FY2025)
Die häufigsten Violation-Typen über alle drei Fiskaljahre (2023–2025) zusammengefasst.

In [ ]:
%%time
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

top10 = df.groupBy("violation_code") \
    .agg(
        f.count("*").alias("count"),
        f.first("violation_description_official").alias("description")
    ) \
    .orderBy(f.desc("count")) \
    .limit(10) \
    .toPandas()

top10_sorted = top10.sort_values("count")

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(top10_sorted["description"], top10_sorted["count"], color="#1f77b4")
ax.set_title("Top 10 Parking Violation Codes FY2023–FY2025", fontsize=13, fontweight="bold")
ax.set_xlabel("Anzahl Violations")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
ax.grid(True, axis="x", alpha=0.4)

x_max = top10_sorted["count"].max()
for bar in bars:
    w = bar.get_width()
    ax.text(w + x_max * 0.005, bar.get_y() + bar.get_height()/2,
            f"{w/1e6:.2f}M", va="center", fontsize=9)

plt.tight_layout()
plt.show()

print(top10[["violation_code", "description", "count"]].to_string(index=False))

### Trend über die Jahre
Verändern sich die häufigsten Violations über FY2023, FY2024 und FY2025?

In [ ]:
%%time
top5_codes = [
    row["violation_code"]
    for row in df.groupBy("violation_code")
                .count()
                .orderBy(f.desc("count"))
                .limit(5)
                .collect()
]
print("Top 5 Codes:", top5_codes)

yearly_trend = df.filter(f.col("violation_code").isin(top5_codes)) \
    .groupBy("fy", "violation_code") \
    .count() \
    .orderBy("violation_code", "fy") \
    .toPandas()

pivot_trend = yearly_trend.pivot(index="fy", columns="violation_code", values="count")

fig, ax = plt.subplots(figsize=(12, 6))
pivot_trend.plot(kind="bar", ax=ax)
ax.set_title("Top 5 Violation Codes pro Fiskaljahr", fontsize=13, fontweight="bold")
ax.set_xlabel("Fiskaljahr")
ax.set_ylabel("Anzahl Violations")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
ax.grid(True, axis="y", alpha=0.4)
ax.legend(title="Violation Code")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### Detailzahlen und Veränderung der Top-5-Codes FY2023–FY2025

Exakte Jahreswerte und prozentuale Entwicklung für alle fünf häufigsten Violation-Typen.

In [ ]:
%%time
summary = df \
    .filter(f.col("violation_code").isin(top5_codes)) \
    .groupBy("fy", "violation_code") \
    .agg(
        f.count("*").alias("count"),
        f.first("violation_description_official").alias("description")
    ) \
    .orderBy("violation_code", "fy") \
    .toPandas()

print(summary.to_string(index=False))

# Prozentuale Veränderung FY2023 → FY2025 für alle Top-5-Codes
print("\nVeränderung FY2023 → FY2025:")
for code in sorted(top5_codes):
    sub = summary[summary["violation_code"] == code].set_index("fy")
    c2023 = sub.loc[2023, "count"]
    c2025 = sub.loc[2025, "count"]
    desc = sub.loc[2023, "description"]
    pct = (c2025 - c2023) / c2023 * 100
    print(f"  Code {code:>2} ({desc[:35]:<35}): {pct:+.1f}%")

### Code 36 - Monatliche Entwicklung nach Fiskaljahr

In [ ]:
%%time
monthly_36 = (df_raw
    .filter("is_complete_fy")
    .filter(f.col("violation_code") == "36")
    .groupBy("fy", "fm")
    .count()
    .orderBy("fy", "fm")
    .toPandas())

pivot_36 = monthly_36.pivot(index="fm", columns="fy", values="count")

fm_labels = {1:"Jul", 2:"Aug", 3:"Sep", 4:"Okt", 5:"Nov", 6:"Dez",
             7:"Jan", 8:"Feb", 9:"Mär", 10:"Apr", 11:"Mai", 12:"Jun"}
pivot_36.index = pivot_36.index.map(fm_labels)

fig, ax = plt.subplots(figsize=(12, 5))
pivot_36.plot(marker="o", ax=ax)
ax.set_title("Code 36 – Monatliche Entwicklung nach Fiskaljahr", 
             fontsize=13, fontweight="bold")
ax.set_xlabel("Monat (Fiskaljahr beginnt im Juli)")
ax.set_ylabel("Anzahl Violations")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.2f}M"))
ax.grid(True, alpha=0.4)
ax.legend(title="Fiskaljahr")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### Fachliche Plausibilisierung der Ergebnis Code 36 Schulzone Tempolimit

Code 36 wird fachlich plausibilisiert, weil er mit Abstand am
häufigsten vorkommt und gleichzeitig den grössten Rückgang aufweist, beides
zusammen macht ihn zum erklärungsbedürftigsten Befund. Die Analyse beschränkt
sich auf Code 36, um den Rahmen des Berichts nicht zu sprengen.

Im monatlichen Chart fällt sofort der Peak im August 2022 (FY2023) auf:
rund 0.77 Mio. Verstösse, deutlich über allen anderen Monaten und Fiskaljahren.
Diese Auffälligkeit war der Ausgangspunkt für eine gezielte Recherche: Warum
genau in diesem Monat ein so markanter Anstieg?

Die Antwort liefert eine dokumentierte Massnahme: Am 1. August 2022 begann NYC
seine 2'000 Schulzonen-Kameras in 750 Schulzonen rund um die Uhr (24/7) zu
betreiben, vorher liefen sie nur werktags zwischen 6:00 und 22:00 Uhr [1]. August
2022 war damit der erste Monat mit voller Kameraauslastung, während Fahrer ihr
Verhalten noch nicht angepasst hatten. Die Kurve sinkt danach kontinuierlich, da
die Fahrer sich vermutlich mit der Zeit an die neuen Blitzer-Zeiten angepasst hatten.
Der monatliche Chart bestätigt diesen Befund direkt aus unseren Daten.

Anmerkung: Der wiederkehrende Rückgang im Februar (alle drei Fiskaljahre) ist
ein saisonales Muster, weniger Tage im Monat und eingeschränktes Fahraufkommen
im Winter. 
Das übergeordnete Muster (Juli hoch -> Februar tief -> Frühling Erholung)
entspricht dem allgemeinen saisonalen Fahraufkommen in NYC. Eine Analyse von
NYC-Verkehrsdaten zeigt, dass der Verkehr im Sommer seinen Höchststand erreicht
und im Januar/Februar auf den Jahrestiefpunkt sinkt [2]. Die Verstosszahlen
spiegeln dieses Muster direkt wider.
Der auffällige Einbruch am Ende von FY2025 (Juni 2025) könnte einen
Reporting-Lag des zuletzt exportierten Dateimonats darstellen und bleibt eine
offene Folgefrage.

[1] New York City Department of Finance. (n.d.). School zone speed camera
    violations. NYC.gov.
    https://www.nyc.gov/site/finance/vehicles/school-zone-camera-violations.page

[2] Gardner, D. R. (n.d.). Seasonal trends in NYC traffic: STL part I.
    gardner.fyi. http://www.gardner.fyi/blog/STL-Part-I/

### Plausibilitätskontrolle: Summe aller Code-Counts 

In [ ]:
%%time
total_sum = (df.groupBy("violation_code")
               .count()
               .agg(f.sum("count").alias("total"))
               .collect()[0]["total"])
total_rows = df.count()

print(f"Summe aller Code-Counts : {total_sum:,}")
print(f"Analysebasis df.count() : {total_rows:,}")
print(f"Differenz               : {total_sum - total_rows:,}")
assert total_sum == total_rows, "✗ Abweichung festgestellt!"

Plausibilitätskontrolle bestanden, keine fehlenden oder doppelten Datensätze.

## Interpretation

- **Code 36 (Schulzone Tempolimit)** dominiert mit 16.77 Mio. Verstössen 
  über alle drei Fiskaljahre und geht von FY2023 auf FY2025 um −24.6 % zurück.
- **Code 21 (No Parking – Street Cleaning)** bleibt konstant auf Rang 2 (−11.9 %).
- **Code 38 (Meter-Beleg)** und **Code 14 (No Standing)** sind weitgehend 
  stabil (+5.0 % bzw. −3.3 %).
- **Code 5 (Bus Lane Violation)** steigt gegen den Trend stark an (+36.6 %).
- Die Rangfolge der fünf häufigsten Typen bleibt über alle drei Jahre stabil.

## Fazit Analyse 1

Die Frage, welche Parking-Violation-Typen am häufigsten vorkommen und wie sie
sich über die Fiskaljahre verändern, lässt sich klar beantworten:

Code 36 (Schulzone Tempolimit) dominiert mit 16.77 Mio. Verstössen über drei
Jahre mit grossem Abstand. Der Rückgang von 24.6 % (FY2023–FY2025) ist keine
Datenfluktuation, sondern eine dokumentierte Verhaltensanpassung. Autofahrer
reagieren auf die seit August 2022 rund um die Uhr aktive Kameraüberwachung in
750 Schulzonen. Bemerkenswert ist der gegenläufige Anstieg bei Bus-Lane-
Verstössen (Code 5, +36.6 %), der zeigt, dass die Stadt gleichzeitig neue
Kontrollschwerpunkte setzt. Die übrigen Top-5-Codes bleiben weitgehend stabil.

Die Daten spiegeln damit weniger eine generelle Ab- oder Zunahme von Verstössen
wider, sondern eine Verschiebung im Fokus der städtischen Verkehrskontrollen.

In [ ]:
spark.stop()